## MLegS Post-Processing Visualization

This notebook is designed to read and visualize 2D field data generated by the `postproc` executable from the MLegS simulation package.

### Functionality:
1.  **Reads Collocation Points**: Loads the physical grid coordinates (`r`, `theta`, `z`) from the `.info` files in the output directory.
2.  **Identifies Data Files**: Scans the output directory for 2D slice data files (e.g., `velR_RTplane_001.dat`, `vorZ_RZplane_010.dat`).
3.  **Parses Filenames**: Extracts metadata from filenames, such as the field type, slice orientation, and snapshot index.
4.  **Loads and Reshapes Data**: Reads the raw data, which is stored as pairs of real and imaginary components, and reshapes it into the correct 2D complex array corresponding to the slice.
5.  **Visualizes Fields**: Creates contour plots for the real part of the selected field data. It handles both R-Theta and R-Z plane visualizations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from ipywidgets import interact, Dropdown, Layout, FloatSlider, Checkbox

## Load data

In [ ]:
def load_collocation_points(directory):
    """Loads r, theta, and z grid points from .info files."""
    try:
        r_pts = np.loadtxt(os.path.join(directory, 'r_colloc_pts.info'))
        th_pts = np.loadtxt(os.path.join(directory, 't_colloc_pts.info'))
        z_pts = np.loadtxt(os.path.join(directory, 'z_colloc_pts.info'))
        print(f"Loaded grid points:")
        print(f"  - Radial (r): {len(r_pts)} points")
        print(f"  - Azimuthal (theta): {len(th_pts)} points")
        print(f"  - Axial (z): {len(z_pts)} points")
        return r_pts, th_pts, z_pts
    except FileNotFoundError as e:
        print(f"Error loading grid files: {e}")
        print("Please ensure 'r_colloc_pts.info', 't_colloc_pts.info', and 'z_colloc_pts.info' are in the output directory.")
        return None, None, None
    
def find_data_files(directory):
    """Finds and parses 2D slice data files in the given directory."""
    # Regex to match filenames like 'velR_RTplane_001.dat' or 'vorZ_RZplane_010.dat'
    # It captures the field name, slice type, and index.
    # It captures the field name, slice type, and index (either 3 digits or 'ini').
    pattern = re.compile(r"^(?P<field>\w+?)_(?P<slice_type>RTplane|RZplane)_(?P<index>\d{3}|ini)\.dat$")
    
    file_info = []
    print(f"\nScanning for data files in: {directory}")
    for filename in sorted(os.listdir(directory)):
        match = pattern.match(filename)
        if match:
            info = match.groupdict()
            info['filename'] = filename
            file_info.append(info)
            
    if not file_info:
        print("No 2D slice data files found. Make sure 'POSTPROCESS%SLICEINT' is not 999 and filenames match the expected pattern.")
    else:
        print(f"Found {len(file_info)} data files.")
        for info in file_info:
            print(f"  - {info['filename']}: Field = {info['field']}, Slice = {info['slice_type']}, Index = {info['index']}")
        
    return file_info

def load_and_reshape_data(filepath, slice_type, grid_shapes):
    """
    Loads data from a file and reshapes it into the correct 2D real-valued array.
    - For RTplane, the data is real across all theta points.
    - For RZplane, the z-direction is periodic. The data is saved for NZ points,
      but the coordinates have NZ+1 points. We pad the data array by copying
      the first z-slice to the end to close the domain for plotting.
      
    Data storage convention:
    - For 2D data: slowest dimension is radial (r), fastest is the other dimension
    - For 3D data: slowest is z, then r, then theta (fastest)
    """
    try:
        # The data is a single line of text with space-separated numbers.
        raw_data = np.loadtxt(filepath).T
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

    # Determine the correct shape based on the slice type
    nr, nth, nz = grid_shapes
    if slice_type == 'RTplane':
        # For RT plane: data layout is nr x (nth-1) with radial as slowest dimension
        # The first nth values are for r=0, next nth for r=1, etc.
        expected_shape = (nth - 1, nr)  # Shape of raw data as stored
        real_data_unpadded = raw_data
        if real_data_unpadded.shape != expected_shape:
            print(f"Error: Data size mismatch for {filepath}.")
            print(f"  - Expected {expected_shape} real values, but found {real_data_unpadded.shape}.")
            print(f"  - Expected shape for storage: {expected_shape}")
            return None
        
        real_data = np.vstack((real_data_unpadded, real_data_unpadded[0:1, :]))  # Pad theta by appending first column

        return real_data.T
        
    elif slice_type == 'RZplane':
        # For RZ plane: data layout is nr x (nz-1) with radial as slowest dimension
        # The first (nz-1) values are for r=0, next (nz-1) for r=1, etc.
        expected_shape = (nz - 1, nr)  # Shape of complex data as stored
        
        if raw_data.size % 2 != 0:
            print(f"Warning: RZ data in {filepath} has an odd number of elements.")
            return None
            
        # Reconstruct complex numbers and take the real part for the slice.
        complex_data = raw_data[::2] + 1j * raw_data[1::2]
        real_data_unpadded = np.real(complex_data)
        if real_data_unpadded.shape != expected_shape:
            print(f"Error: Data size mismatch for {filepath}.")
            print(f"  - Expected {expected_shape} real values, but found {real_data_unpadded.shape}.")
            print(f"  - Expected shape for storage: {expected_shape}")
            return None
        
        real_data = np.vstack((real_data_unpadded, real_data_unpadded[0:1, :])) # Pad z by appending first column

        return real_data.T # Return padded data with shape (nr, nz)
    else:
        return None


## Visualization

In [ ]:
def plot_rt_slice(data, r_coords, th_coords, title, r_min = 0.0, r_max = 5.0, use_log_scale=True, 
                  mask_low_intensity=False, intensity_threshold_orders=2):
    """Creates a polar contour plot for an R-Theta slice.
    
    Parameters:
    -----------
    mask_low_intensity : bool, default False
        If True, values below the intensity threshold will be shown as white (masked)
    intensity_threshold_orders : float, default 2
        Number of orders of magnitude below max value to mask (e.g., 2 means mask values < max/100)
    """
    if data is None:
        return

    # Filter data to only include points within r_max
    r_mask = (r_coords <= r_max) & (r_coords >= r_min)
    r_filtered = r_coords[r_mask]
    data_filtered = data[r_mask, :]

    # Check if data is all zeros
    max_abs_val = np.max(np.abs(data_filtered))
    if max_abs_val < 1e-16:
        use_log_scale = False  # Force linear scale for zero data

    # Apply intensity masking if requested
    if mask_low_intensity and max_abs_val >= 1e-16:
        intensity_threshold = max_abs_val / (10 ** intensity_threshold_orders)
        mask = np.abs(data_filtered) < intensity_threshold
        data_filtered_masked = np.ma.masked_where(mask, data_filtered)
    else:
        data_filtered_masked = data_filtered

    # Create a meshgrid for polar plot
    R, TH = np.meshgrid(r_filtered, th_coords)

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})
    
    if use_log_scale and max_abs_val >= 1e-16:
        # Apply symmetric logarithmic transformation
        # For values with same sign, use sign(x) * log10(1 + |x|/threshold)
        threshold = max_abs_val * 1e-6  # Threshold to avoid log(0)
        
        # Transform data: sign(x) * log10(1 + |x|/threshold)
        data_log = np.sign(data_filtered_masked.T) * np.log10(1 + np.abs(data_filtered_masked.T) / threshold)
        
        # Set symmetric color limits for log-transformed data
        max_log_val = np.max(np.abs(data_log))
        vmin, vmax = -max_log_val, max_log_val
        
        # Use a colormap with white for masked values
        cmap = plt.cm.RdBu_r
        if mask_low_intensity:
            cmap = plt.cm.RdBu_r.copy()
            cmap.set_bad(color='white')
        
        contour = ax.pcolormesh(TH, R, data_log, shading='auto', cmap=cmap, 
                               vmin=vmin, vmax=vmax)
        
        # Create custom colorbar with original values
        cbar = fig.colorbar(contour, ax=ax, orientation='vertical', label='Field Value')
        
        # Create custom tick labels showing original values
        n_ticks = 7
        log_ticks = np.linspace(vmin, vmax, n_ticks)
        original_ticks = np.sign(log_ticks) * threshold * (10**np.abs(log_ticks) - 1)
        
        cbar.set_ticks(log_ticks)
        cbar.set_ticklabels([f'{val:.2e}' for val in original_ticks])
        
    else:
        # Standard linear color scale
        if max_abs_val < 1e-16:
            # For all-zero data, use small symmetric limits
            vmin, vmax = -1e-16, 1e-16
        else:
            vmin, vmax = -max_abs_val, max_abs_val
        
        # Use a colormap with white for masked values
        cmap = plt.cm.RdBu_r
        if mask_low_intensity:
            cmap = plt.cm.RdBu_r.copy()
            cmap.set_bad(color='white')
        
        contour = ax.pcolormesh(TH, R, data_filtered_masked.T, shading='auto', cmap=cmap, 
                               vmin=vmin, vmax=vmax)
        
        fig.colorbar(contour, ax=ax, orientation='vertical', label='Field Value')
    
    ax.set_title(title, va='bottom', fontsize=14)
    ax.set_ylim(0, r_max) # Set radial limit to r_max
    ax.set_xlabel('$\\theta$')
    ax.set_ylabel('$r$', labelpad=20)
    plt.show()

def plot_rz_slice(data, r_coords, z_coords, title, r_min = 0.0, r_max = 5.0, use_log_scale=True,
                  mask_low_intensity=False, intensity_threshold_orders=2):
    """Creates a Cartesian contour plot for an R-Z slice.
    
    Parameters:
    -----------
    mask_low_intensity : bool, default False
        If True, values below the intensity threshold will be shown as white (masked)
    intensity_threshold_orders : float, default 2
        Number of orders of magnitude below max value to mask (e.g., 2 means mask values < max/100)
    """
    if data is None:
        return

    # Filter data to only include points within r_max
    r_mask = (r_coords <= r_max) & (r_coords >= r_min)
    r_filtered = r_coords[r_mask]
    data_filtered = data[r_mask, :]

    # Check if data is all zeros
    max_abs_val = np.max(np.abs(data_filtered))
    if max_abs_val < 1e-16:
        use_log_scale = False  # Force linear scale for zero data

    # Apply intensity masking if requested
    if mask_low_intensity and max_abs_val >= 1e-16:
        intensity_threshold = max_abs_val / (10 ** intensity_threshold_orders)
        mask = np.abs(data_filtered) < intensity_threshold
        data_filtered_masked = np.ma.masked_where(mask, data_filtered)
    else:
        data_filtered_masked = data_filtered

    # Calculate ZLEN for normalization
    ZLEN = np.max(z_coords) - np.min(z_coords)
    z_normalized = z_coords / ZLEN

    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Create a meshgrid for the plot with r as x-axis and normalized z as y-axis
    R, Z = np.meshgrid(r_filtered, z_normalized)

    if use_log_scale and max_abs_val >= 1e-16:
        # Apply symmetric logarithmic transformation
        threshold = max_abs_val * 1e-6  # Threshold to avoid log(0)
        
        # Transform data: sign(x) * log10(1 + |x|/threshold)
        data_log = np.sign(data_filtered_masked.T) * np.log10(1 + np.abs(data_filtered_masked.T) / threshold)
        
        # Set symmetric color limits for log-transformed data
        max_log_val = np.max(np.abs(data_log))
        vmin, vmax = -max_log_val, max_log_val
        
        # Use a colormap with white for masked values
        cmap = plt.cm.RdBu_r
        if mask_low_intensity:
            cmap = plt.cm.RdBu_r.copy()
            cmap.set_bad(color='white')
        
        contour = ax.pcolormesh(R, Z, data_log, shading='auto', cmap=cmap,
                               vmin=vmin, vmax=vmax)
        
        # Create custom colorbar with original values
        cbar = fig.colorbar(contour, ax=ax, label='Field Value')
        
        # Create custom tick labels showing original values
        n_ticks = 7
        log_ticks = np.linspace(vmin, vmax, n_ticks)
        original_ticks = np.sign(log_ticks) * threshold * (10**np.abs(log_ticks) - 1)
        
        cbar.set_ticks(log_ticks)
        cbar.set_ticklabels([f'{val:.2e}' for val in original_ticks])
        
    else:
        # Standard linear color scale
        if max_abs_val < 1e-16:
            # For all-zero data, use small symmetric limits
            vmin, vmax = -1e-16, 1e-16
        else:
            vmin, vmax = -max_abs_val, max_abs_val
        
        # Use a colormap with white for masked values
        cmap = plt.cm.RdBu_r
        if mask_low_intensity:
            cmap = plt.cm.RdBu_r.copy()
            cmap.set_bad(color='white')
        
        contour = ax.pcolormesh(R, Z, data_filtered_masked.T, shading='auto', cmap=cmap,
                               vmin=vmin, vmax=vmax)
        
        fig.colorbar(contour, ax=ax, label='Field Value')
    
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Radial Coordinate (r)')
    ax.set_ylabel('Axial Coordinate (z/ZLEN)')
    ax.set_xlim(0, r_max) # Set radial limit to r_max
    plt.show()

def interactive_plotter(file_to_plot, r_min = 0.0, r_max=5.0, use_log_scale=False, rt_use_log_scale=False, rz_use_log_scale=False,
                       mask_low_intensity=False, intensity_threshold_orders=2):
    """Main function driven by the ipywidgets interact decorator.
    
    Parameters:
    -----------
    mask_low_intensity : bool, default False
        If True, values below the intensity threshold will be shown as white (masked)
    intensity_threshold_orders : float, default 2
        Number of orders of magnitude below max value to mask (e.g., 2 means mask values < max/100)
    """
    if not file_to_plot or not all((r_pts is not None, th_pts is not None, z_pts is not None)):
        print("Cannot plot. Check that data and grid files were loaded correctly.")
        return
    
    # Find the corresponding info dictionary
    info = next((item for item in data_files if item['filename'] == file_to_plot), None)
    if not info:
        print(f"Could not find info for {file_to_plot}")
        return

    filepath = os.path.join(output_dir, info['filename'])
    grid_shapes = (len(r_pts), len(th_pts), len(z_pts))

    # Load and process the data
    data = load_and_reshape_data(filepath, info['slice_type'], grid_shapes)

    if data is None:
        print(f"Failed to load or process data for {info['filename']}.")
        return

    # Generate title and call the correct plotting function
    title = f"Field: {info['field']} | Slice: {info['slice_type']} | Index: {info['index']}"
    
    if info['slice_type'] == 'RTplane':
        plot_rt_slice(data, r_pts, th_pts, title, r_min=r_min, r_max=r_max, 
                     use_log_scale=rt_use_log_scale or use_log_scale,
                     mask_low_intensity=mask_low_intensity, 
                     intensity_threshold_orders=intensity_threshold_orders)
    elif info['slice_type'] == 'RZplane':
        plot_rz_slice(data, r_pts, z_pts, title, r_min=r_min, r_max=r_max, 
                     use_log_scale=rz_use_log_scale or use_log_scale,
                     mask_low_intensity=mask_low_intensity,
                     intensity_threshold_orders=intensity_threshold_orders)

## Analysis

In [ ]:
output_dir = '../../output/'
r_pts, th_pts, z_pts = load_collocation_points(output_dir)

data_files = find_data_files(output_dir)
info = data_files[0]
filepath = os.path.join(output_dir, info['filename'])
grid_shapes = (len(r_pts), len(th_pts), len(z_pts))

# Load and process the data
data = load_and_reshape_data(filepath, info['slice_type'], grid_shapes)
print(data.shape)

In [ ]:
# Create interactive widgets with better layout
if data_files:
    from ipywidgets import HBox, VBox, Label, interactive_output
    
    # Create widgets
    file_dropdown = Dropdown(options=[f['filename'] for f in data_files], description='File:')
    r_min_slider = FloatSlider(value=2.0, min=0.0, max=90.0, step=0.1, description='r_min:', continuous_update=False)
    r_max_slider = FloatSlider(value=30.0, min=1.0, max=90.0, step=0.1, description='r_max:', continuous_update=False)
    log_both_check = Checkbox(value=False, description='Log (both)')
    log_rt_check = Checkbox(value=True, description='Log RT')
    log_rz_check = Checkbox(value=True, description='Log RZ')
    mask_check = Checkbox(value=False, description='Mask low intensity')
    threshold_slider = FloatSlider(value=2.0, min=1.0, max=6.0, step=0.5, description='Orders:', continuous_update=False)
    
    # Organize layout
    ui = VBox([
        file_dropdown,
        HBox([r_min_slider, r_max_slider]),
        HBox([log_both_check, log_rt_check, log_rz_check]),
        HBox([mask_check, threshold_slider])
    ])
    
    # Connect to function
    out = interactive_output(interactive_plotter, {
        'file_to_plot': file_dropdown,
        'r_min': r_min_slider,
        'r_max': r_max_slider,
        'use_log_scale': log_both_check,
        'rt_use_log_scale': log_rt_check,
        'rz_use_log_scale': log_rz_check,
        'mask_low_intensity': mask_check,
        'intensity_threshold_orders': threshold_slider
    })
    
    display(ui, out)
else:
    print("\nNo files to display. Run the cells above to scan for data.")